# Analyse TF-IDF Artifacts

This notebook analyzes token TF-IDF artifacts from each scope's working directory, `artifacts/tokenizers/work/<scope>/`, where `company_tokenize.scope_directory_files` names the statistics file (parquet, or its csv).

It is independent of optimize artifacts, includes `global`, and excludes `bert`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401
from company_tokenize import ScopeDirectoryFiles, scope_directory_files

from workspace.artifact_layout import tokenizer_artifact_root, tokenizer_scope_dir
from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

SYSTEMS = []  # Leave empty to auto-discover TF-IDF artifact systems.
EXCLUDED_SYSTEMS = {"bert"}
# Every scope's working directory sits beneath this one, named for its system
# code, or `global` for the scope with none.
SCOPES_ROOT = tokenizer_artifact_root(ROOTS)
GLOBAL_SCOPE = "global"


def scope_files(system: str) -> ScopeDirectoryFiles:
    """The corpus files of one scope, named by the package that writes them."""
    return scope_directory_files(
        tokenizer_scope_dir(ROOTS, system=None if system == GLOBAL_SCOPE else system)
    )


TFIDF_BUCKET_COUNT = 20


def resolve_tfidf_systems() -> list[str]:
    if SYSTEMS:
        return [s for s in SYSTEMS if s not in EXCLUDED_SYSTEMS]
    if not SCOPES_ROOT.exists():
        raise FileNotFoundError(f"No tokenizer scope directory under: {SCOPES_ROOT}")

    discovered: list[str] = []
    for path in sorted(SCOPES_ROOT.iterdir()):
        if not path.is_dir() or path.name in EXCLUDED_SYSTEMS:
            continue
        files = scope_files(path.name)
        if files.tfidf_stats.exists() or files.tfidf_stats_csv.exists():
            discovered.append(path.name)
    return discovered


def _load_tfidf_artifact_frame(system: str) -> pl.DataFrame:
    files = scope_files(system)
    parquet_path = files.tfidf_stats
    csv_path = files.tfidf_stats_csv

    if parquet_path.exists():
        frame = pl.read_parquet(parquet_path)
    elif csv_path.exists():
        frame = pl.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"Missing TF-IDF artifact for system={system}")

    required_cols = {"document_frequency_pct", "idf"}
    missing_cols = required_cols.difference(frame.columns)
    if missing_cols:
        raise ValueError(f"system={system} missing columns: {sorted(missing_cols)}")

    return frame.select(
        [
            pl.col("document_frequency_pct").cast(pl.Float64, strict=False),
            pl.col("idf").cast(pl.Float64, strict=False),
        ]
    ).drop_nulls()


SYSTEMS = resolve_tfidf_systems()
if not SYSTEMS:
    raise FileNotFoundError(
        f"No tf-idf statistics in any scope directory under {SCOPES_ROOT}. Make "
        "them with: uv run python scripts/generate_noise_words.py "
        "--system <code>"
    )

print("TF-IDF analysis systems:", ", ".join(SYSTEMS))
display(
    pd.DataFrame(
        {"system": SYSTEMS, "includes_global": [s == "global" for s in SYSTEMS]}
    )
)

TF-IDF analysis systems: gb, global, ie


,system,includes_global
0,gb,False
1,global,True
2,ie,False


In [4]:
summary_rows = []
for system in SYSTEMS:
    frame = _load_tfidf_artifact_frame(system)
    stats = frame.select(
        [
            pl.len().alias("token_rows"),
            pl.col("document_frequency_pct").min().alias("min_df_pct"),
            pl.col("document_frequency_pct").max().alias("max_df_pct"),
            pl.col("idf").min().alias("min_idf"),
            pl.col("idf").max().alias("max_idf"),
        ]
    ).to_dicts()[0]
    stats["system"] = system
    summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows).sort_values("system")
display(
    summary_df[
        ["system", "token_rows", "min_df_pct", "max_df_pct", "min_idf", "max_idf"]
    ]
)

# Build DF% buckets on a log scale to avoid collapsing almost all unique tokens into bucket 0.
df_max = float(summary_df["max_df_pct"].max())
df_min_positive = float(summary_df["min_df_pct"][summary_df["min_df_pct"] > 0].min())

# Keep IDF buckets linear; this dimension is already less compressed.
idf_min = float(summary_df["min_idf"].min())
idf_max = float(summary_df["max_idf"].max())
idf_span = max(idf_max - idf_min, 1e-12)

df_edges = np.geomspace(
    df_min_positive, max(df_max, df_min_positive), TFIDF_BUCKET_COUNT + 1
)
idf_edges = np.linspace(idf_min, idf_max, TFIDF_BUCKET_COUNT + 1)

df_rows = []
idf_rows = []

for system in SYSTEMS:
    frame = _load_tfidf_artifact_frame(system)
    df_values = frame.get_column("document_frequency_pct").to_numpy()
    idf_values = frame.get_column("idf").to_numpy()

    # Unique-token count per bucket (one row in stats == one unique token).
    df_bucket_idx = np.digitize(df_values, df_edges[1:], right=False)
    df_bucket_idx = np.clip(df_bucket_idx, 0, TFIDF_BUCKET_COUNT - 1)
    df_token_counts = np.bincount(df_bucket_idx, minlength=TFIDF_BUCKET_COUNT)
    df_mass_counts = np.bincount(
        df_bucket_idx, weights=df_values, minlength=TFIDF_BUCKET_COUNT
    )

    idf_bucket_idx = np.digitize(idf_values, idf_edges[1:], right=False)
    idf_bucket_idx = np.clip(idf_bucket_idx, 0, TFIDF_BUCKET_COUNT - 1)
    idf_token_counts = np.bincount(idf_bucket_idx, minlength=TFIDF_BUCKET_COUNT)

    total_unique_tokens = max(int(df_token_counts.sum()), 1)
    total_df_mass = max(float(df_mass_counts.sum()), 1e-12)

    for idx in range(TFIDF_BUCKET_COUNT):
        token_count = int(df_token_counts[idx])
        if token_count == 0:
            continue
        start = float(df_edges[idx])
        end = float(df_edges[idx + 1])
        df_mass = float(df_mass_counts[idx])
        df_rows.append(
            {
                "system": system,
                "bucket_idx": idx,
                "bucket_start": start,
                "bucket_end": end,
                "bucket_label": f"[{start:.8f}, {end:.8f})",
                "token_count": token_count,
                "token_unique_share": token_count / total_unique_tokens,
                "df_mass": df_mass,
                "df_mass_share": df_mass / total_df_mass,
            }
        )

    for idx in range(TFIDF_BUCKET_COUNT):
        token_count = int(idf_token_counts[idx])
        if token_count == 0:
            continue
        start = float(idf_edges[idx])
        end = float(idf_edges[idx + 1])
        idf_rows.append(
            {
                "system": system,
                "bucket_idx": idx,
                "bucket_start": start,
                "bucket_end": end,
                "bucket_label": f"[{start:.4f}, {end:.4f})",
                "token_count": token_count,
                "token_unique_share": token_count / total_unique_tokens,
            }
        )

df_bucket_counts = pd.DataFrame(df_rows).sort_values(["system", "bucket_idx"])
idf_bucket_counts = pd.DataFrame(idf_rows).sort_values(["system", "bucket_idx"])

df_bucket_counts["cumulative_unique_share"] = df_bucket_counts.groupby("system")[
    "token_unique_share"
].cumsum()
df_bucket_counts["cumulative_df_mass_share"] = df_bucket_counts.groupby("system")[
    "df_mass_share"
].cumsum()
idf_bucket_counts["cumulative_unique_share"] = idf_bucket_counts.groupby("system")[
    "token_unique_share"
].cumsum()

print("DF buckets are log-scaled; token_count is unique-token count per bucket.")
display(df_bucket_counts.head(40))
display(idf_bucket_counts.head(40))

,system,token_rows,min_df_pct,max_df_pct,min_idf,max_idf
0,gb,1295457,1.754917e-07,0.953088,1.048048,15.862527
1,global,4723389,3.575329e-08,0.241460,2.421052,17.453476
2,ie,297657,1.222844e-06,0.925171,1.077777,13.921186


DF buckets are log-scaled; token_count is unique-token count per bucket.


,system,bucket_idx,bucket_start,bucket_end,bucket_label,token_count,token_unique_share,df_mass,df_mass_share,cumulative_unique_share,cumulative_df_mass_share
0,gb,1,8.406339e-08,1.976504e-07,"[0.00000008, 0.00000020)",970198,7.489234e-01,0.170262,0.048804,0.748923,0.048804
1,gb,2,1.976504e-07,4.647171e-07,"[0.00000020, 0.00000046)",129953,1.003144e-01,0.045611,0.013074,0.849238,0.061878
2,gb,3,4.647171e-07,1.092646e-06,"[0.00000046, 0.00000109)",107726,8.315675e-02,0.073791,0.021151,0.932395,0.083029
3,gb,4,1.092646e-06,2.569037e-06,"[0.00000109, 0.00000257)",37969,2.930935e-02,0.063651,0.018245,0.961704,0.101274
4,gb,5,2.569037e-06,6.040337e-06,"[0.00000257, 0.00000604)",22695,1.751891e-02,0.088399,0.025339,0.979223,0.126612
5,gb,6,6.040337e-06,1.420208e-05,"[0.00000604, 0.00001420)",12820,9.896122e-03,0.117537,0.033691,0.989119,0.160303
6,gb,7,1.420208e-05,3.339204e-05,"[0.00001420, 0.00003339)",7151,5.520060e-03,0.152238,0.043637,0.994639,0.203940
7,gb,8,3.339204e-05,7.851159e-05,"[0.00003339, 0.00007851)",3536,2.729539e-03,0.177653,0.050922,0.997368,0.254863
8,gb,9,7.851159e-05,1.845970e-04,"[0.00007851, 0.00018460)",1836,1.417260e-03,0.218718,0.062693,0.998786,0.317556
9,gb,10,1.845970e-04,4.340257e-04,"[0.00018460, 0.00043403)",925,7.140337e-04,0.255439,0.073219,0.999500,0.390775


,system,bucket_idx,bucket_start,bucket_end,bucket_label,token_count,token_unique_share,cumulative_unique_share
0,gb,0,1.048048,1.868319,"[1.0480, 1.8683)",1,7.719284e-07,7.719284e-07
1,gb,3,3.508862,4.329133,"[3.5089, 4.3291)",1,7.719284e-07,1.543857e-06
2,gb,4,4.329133,5.149405,"[4.3291, 5.1494)",9,6.947355e-06,8.491212e-06
3,gb,5,5.149405,5.969676,"[5.1494, 5.9697)",12,9.263140e-06,1.775435e-05
4,gb,6,5.969676,6.789948,"[5.9697, 6.7899)",48,3.705256e-05,5.480691e-05
5,gb,7,6.789948,7.610219,"[6.7899, 7.6102)",137,1.057542e-04,1.605611e-04
6,gb,8,7.610219,8.430491,"[7.6102, 8.4305)",267,2.061049e-04,3.666660e-04
7,gb,9,8.430491,9.250762,"[8.4305, 9.2508)",622,4.801394e-04,8.468054e-04
8,gb,10,9.250762,10.071033,"[9.2508, 10.0710)",1355,1.045963e-03,1.892768e-03
9,gb,11,10.071033,10.891305,"[10.0710, 10.8913)",2472,1.908207e-03,3.800975e-03


In [5]:
def safe_show(fig):
    try:
        fig.show()
    except Exception as exc:  # noqa: BLE001 -- a renderer that cannot show a figure must not stop the notebook
        title = (
            fig.layout.title.text
            if fig.layout and fig.layout.title
            else "Untitled figure"
        )
        print(f"Skipping interactive render for '{title}': {exc}")


ordered_systems = [s for s in SYSTEMS if s in df_bucket_counts["system"].unique()]

# Primary view: unique-token counts per bucket.
fig_df = px.bar(
    df_bucket_counts,
    x="bucket_start",
    y="token_count",
    color="system",
    facet_col="system",
    facet_col_wrap=3,
    category_orders={"system": ordered_systems},
    hover_data=["bucket_label", "token_unique_share", "df_mass_share"],
    title="Unique token count by DF% bucket (log-scaled buckets)",
    labels={"bucket_start": "DF% bucket start", "token_count": "Unique token count"},
)
fig_df.update_layout(showlegend=False)
safe_show(fig_df)

fig_idf = px.bar(
    idf_bucket_counts,
    x="bucket_start",
    y="token_count",
    color="system",
    facet_col="system",
    facet_col_wrap=3,
    category_orders={"system": ordered_systems},
    hover_data=["bucket_label", "token_unique_share"],
    title="Unique token count by IDF bucket",
    labels={"bucket_start": "IDF bucket start", "token_count": "Unique token count"},
)
fig_idf.update_layout(showlegend=False)
safe_show(fig_idf)

# Secondary diagnostics: cumulative curves shown separately from count bars.
fig_df_cum = px.line(
    df_bucket_counts,
    x="bucket_start",
    y="cumulative_unique_share",
    color="system",
    facet_col="system",
    facet_col_wrap=3,
    category_orders={"system": ordered_systems},
    title="Cumulative unique-token share by DF% bucket",
    labels={
        "bucket_start": "DF% bucket start",
        "cumulative_unique_share": "Cumulative unique-token share",
    },
)
fig_df_cum.update_layout(showlegend=False)
safe_show(fig_df_cum)

fig_df_mass_cum = px.line(
    df_bucket_counts,
    x="bucket_start",
    y="cumulative_df_mass_share",
    color="system",
    facet_col="system",
    facet_col_wrap=3,
    category_orders={"system": ordered_systems},
    title="Cumulative DF-mass share by DF% bucket",
    labels={
        "bucket_start": "DF% bucket start",
        "cumulative_df_mass_share": "Cumulative DF-mass share",
    },
)
fig_df_mass_cum.update_layout(showlegend=False)
safe_show(fig_df_mass_cum)

boundary_rows = []
for system in SYSTEMS:
    noise_words_path = scope_files(system).noise_words
    if not noise_words_path.exists():
        continue
    payload = json.loads(noise_words_path.read_text(encoding="utf-8"))
    for profile_name, profile_payload in payload.get("profiles", {}).items():
        selection = profile_payload.get("selection", {})
        boundary_rows.append(
            {
                "system": system,
                "profile": profile_name,
                "token_count_actual": selection.get("token_count_actual"),
                "token_count_target": selection.get("token_count_target"),
                "cumulative_df_mass_target": selection.get("cumulative_df_mass_target"),
                "cumulative_df_mass_actual": selection.get("cumulative_df_mass_actual"),
                "cumulative_df_pct_cutoff": selection.get(
                    "cumulative_doc_frequency_pct_cutoff"
                ),
                "cumulative_idf_cutoff": selection.get("cumulative_idf_cutoff"),
            }
        )

boundary_df = pd.DataFrame(boundary_rows)
if boundary_df.empty:
    print("No noise-word boundary metadata found.")
else:
    display(boundary_df.sort_values(["system", "profile"]))

,system,profile,token_count_actual,token_count_target,cumulative_df_mass_target,cumulative_df_mass_actual,cumulative_df_pct_cutoff,cumulative_idf_cutoff
2,gb,aggressive,25,25,NaN,NaN,0.004821,6.334788
1,gb,balanced,10,10,NaN,NaN,0.020704,4.877412
0,gb,strict,1,1,NaN,NaN,0.953088,1.048048
5,global,aggressive,25,25,0.08,0.208939,0.005731,6.161777
4,global,balanced,10,10,0.03,0.332396,0.016392,5.110976
3,global,strict,1,1,0.01,0.241460,0.241460,2.421052
8,ie,aggressive,25,25,NaN,NaN,0.005686,6.169495
7,ie,balanced,10,10,NaN,NaN,0.019803,4.921875
6,ie,strict,1,1,NaN,NaN,0.925171,1.077777


## Pooled noise-word candidates

Any rule-named sibling of the global directory's `noise_words.json` (for example `noise_words.equal.json`, written by `scripts/generate_noise_words.py --systems ... --pooling equal`) is loaded here and compared per profile against one system's own list, so a pooled candidate can be inspected before any run uses it. Nothing here is promoted to the packaged default -- that waits on separate blocking evidence.

In [ ]:
POOLED_NOISE_WORD_COMPARISON_SYSTEM = (
    "ie"  # change to compare a different system's own list
)


def _load_noise_word_profile_tokens(payload: dict, profile_name: str) -> list[str]:
    profile_payload = payload.get("profiles", {}).get(profile_name, {})
    tokens = profile_payload.get("tokens")
    return list(tokens) if isinstance(tokens, list) else []


def discover_pooled_noise_word_candidates() -> dict[str, Path]:
    """Rule-named siblings of the global directory's noise_words.json.

    Matches ``noise_words.<rule>.json`` beside the global scope's ``noise_words.json``
    (for example ``noise_words.equal.json``, ``noise_words.count.json``);
    the un-suffixed file itself is the packaged default and is excluded.
    """
    global_dir = scope_files(GLOBAL_SCOPE).directory
    if not global_dir.exists():
        return {}

    candidates: dict[str, Path] = {}
    for candidate_path in sorted(global_dir.glob("noise_words.*.json")):
        rule = (
            candidate_path.stem.split(".", 1)[1]
            if "." in candidate_path.stem
            else candidate_path.stem
        )
        candidates[rule] = candidate_path
    return candidates


pooled_candidate_paths = discover_pooled_noise_word_candidates()
if not pooled_candidate_paths:
    print("No pooled noise-word candidate files found beside the global list.")
else:
    own_noise_words_path = scope_files(POOLED_NOISE_WORD_COMPARISON_SYSTEM).noise_words
    own_payload = (
        json.loads(own_noise_words_path.read_text(encoding="utf-8"))
        if own_noise_words_path.exists()
        else {}
    )

    comparison_rows = []
    for rule, candidate_path in pooled_candidate_paths.items():
        pooled_payload = json.loads(candidate_path.read_text(encoding="utf-8"))
        for profile_name in ("strict", "balanced", "aggressive"):
            pooled_tokens = set(
                _load_noise_word_profile_tokens(pooled_payload, profile_name)
            )
            own_tokens = set(_load_noise_word_profile_tokens(own_payload, profile_name))
            comparison_rows.append(
                {
                    "pooling_rule": rule,
                    "profile": profile_name,
                    "pooled_token_count": len(pooled_tokens),
                    f"{POOLED_NOISE_WORD_COMPARISON_SYSTEM}_token_count": len(
                        own_tokens
                    ),
                    "shared": len(pooled_tokens & own_tokens),
                    "pooled_only": sorted(pooled_tokens - own_tokens),
                    f"{POOLED_NOISE_WORD_COMPARISON_SYSTEM}_only": sorted(
                        own_tokens - pooled_tokens
                    ),
                }
            )

    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)